In [11]:
# Notebook bootstrap: define dependencies, fixtures, and implementation before tests.
from __future__ import annotations

import io
import unittest
from abc import ABC, abstractmethod
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any, Mapping, Sequence

import yaml as pyyaml

SAMPLE_YAML = '''
name: demo
enabled: true
count: 3
items:
  - one
  - two
  - three
'''
VALID_DOC = {
    'name': 'demo',
    'enabled': True,
    'count': 3,
    'items': ['one', 'two', 'three'],
}
INVALID_YAML = 'name: [broken'


class YamlContract(ABC):
    @classmethod
    @abstractmethod
    def parse(cls, source: str, *, schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def load(self, source: Any, *, schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def dump(self, value: Any, *, sort_keys: bool = True, default_flow_style: bool = False) -> str:
        raise NotImplementedError

    @abstractmethod
    def read(self, path: str | Path, *, encoding: str = 'utf-8', schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def write(self, path: str | Path, value: Any, *, encoding: str = 'utf-8', sort_keys: bool = True) -> None:
        raise NotImplementedError

    @abstractmethod
    def validate(self, value: Any) -> Any:
        raise NotImplementedError


class YamlError(ValueError):
    """Raised when YAML parsing, validation, or I/O fails."""


class yaml(YamlContract):
    def __init__(self, *, schema: str = 'safe', encoding: str = 'utf-8', strict: bool = True):
        self.schema = schema
        self.encoding = encoding
        self.strict = strict

    @classmethod
    def parse(cls, source: str, *, schema: str = 'safe') -> Any:
        if not isinstance(source, str):
            raise TypeError('source must be a string')
        if not source.strip():
            return None
        try:
            if schema == 'safe':
                return pyyaml.safe_load(source)
            if schema == 'full':
                return pyyaml.load(source, Loader=pyyaml.FullLoader)
            raise ValueError(f'Unsupported schema: {schema!r}')
        except pyyaml.YAMLError as exc:
            raise YamlError(f'Unable to parse YAML: {exc}') from exc

    def load(self, source: Any, *, schema: str = 'safe') -> Any:
        if isinstance(source, (str, bytes)):
            text = source.decode(self.encoding) if isinstance(source, bytes) else source
            return self.parse(text, schema=schema)
        if hasattr(source, 'read'):
            content = source.read()
            if isinstance(content, bytes):
                content = content.decode(self.encoding)
            return self.parse(content, schema=schema)
        if source is None:
            return None
        raise TypeError('load() expects a YAML string, bytes, stream, or None')

    def dump(self, value: Any, *, sort_keys: bool = True, default_flow_style: bool = False) -> str:
        validated = self.validate(value)
        return pyyaml.safe_dump(
            validated,
            sort_keys=sort_keys,
            default_flow_style=default_flow_style,
        )

    def read(self, path: str | Path, *, encoding: str = 'utf-8', schema: str = 'safe') -> Any:
        file_path = Path(path)
        if not file_path.exists():
            raise FileNotFoundError(f'YAML file does not exist: {file_path}')
        return self.load(file_path.read_text(encoding=encoding), schema=schema)

    def write(self, path: str | Path, value: Any, *, encoding: str = 'utf-8', sort_keys: bool = True) -> None:
        file_path = Path(path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(self.dump(value, sort_keys=sort_keys), encoding=encoding)

    def validate(self, value: Any) -> Any:
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        if isinstance(value, Mapping):
            normalized = {}
            for key, item in value.items():
                if not isinstance(key, (str, int)):
                    raise TypeError('Mapping keys must be strings or ints')
                normalized[key] = self.validate(item)
            return normalized
        if isinstance(value, Sequence) and not isinstance(value, (str, bytes, bytearray)):
            return [self.validate(item) for item in value]
        if isinstance(value, set):
            return [self.validate(item) for item in sorted(value, key=repr)]
        raise TypeError(f'Unsupported YAML value type: {type(value)!r}')


example = yaml()
print('Notebook prerequisites ready; unresolved abstract methods:', yaml.__abstractmethods__)


Notebook prerequisites ready; unresolved abstract methods: frozenset()


## 8. Run Tests and Validate Behavior in VS Code

This final section executes the test suite in the notebook environment and confirms that the abstract contract is satisfied, the class is concrete, and the YAML behavior is useful in real use cases.

In [12]:
import unittest
from tempfile import TemporaryDirectory


class TestYamlImplementation(unittest.TestCase):
    def setUp(self):
        self.codec = yaml()

    def test_parse_valid_yaml(self):
        parsed = self.codec.parse(SAMPLE_YAML)
        self.assertEqual(parsed['name'], 'demo')
        self.assertTrue(parsed['enabled'])
        self.assertEqual(parsed['count'], 3)

    def test_parse_invalid_yaml_raises(self):
        with self.assertRaises(YamlError):
            self.codec.parse(INVALID_YAML)

    def test_load_from_string_and_bytes(self):
        self.assertEqual(self.codec.load(SAMPLE_YAML), self.codec.parse(SAMPLE_YAML))
        self.assertEqual(self.codec.load(SAMPLE_YAML.encode('utf-8')), self.codec.parse(SAMPLE_YAML))

    def test_load_from_stream(self):
        stream = io.StringIO(SAMPLE_YAML)
        self.assertEqual(self.codec.load(stream), self.codec.parse(SAMPLE_YAML))

    def test_dump_and_round_trip(self):
        rendered = self.codec.dump(VALID_DOC)
        self.assertIn('name: demo', rendered)
        round_trip = self.codec.parse(rendered)
        self.assertEqual(round_trip, VALID_DOC)

    def test_write_read_round_trip(self):
        with TemporaryDirectory() as tmp:
            path = Path(tmp) / 'doc.yaml'
            self.codec.write(path, VALID_DOC)
            self.assertEqual(self.codec.read(path), VALID_DOC)

    def test_validate_rejects_unsupported_type(self):
        with self.assertRaises(TypeError):
            self.codec.validate({'unsupported': object()})

    def test_class_is_concrete(self):
        self.assertFalse(getattr(yaml, '__abstractmethods__', None))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestYamlImplementation)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print(f'Passed={result.wasSuccessful()}')


test_class_is_concrete (__main__.TestYamlImplementation.test_class_is_concrete) ... ok
test_dump_and_round_trip (__main__.TestYamlImplementation.test_dump_and_round_trip) ... ok
test_load_from_stream (__main__.TestYamlImplementation.test_load_from_stream) ... ok
test_load_from_string_and_bytes (__main__.TestYamlImplementation.test_load_from_string_and_bytes) ... ok
test_parse_invalid_yaml_raises (__main__.TestYamlImplementation.test_parse_invalid_yaml_raises) ... ok
test_parse_valid_yaml (__main__.TestYamlImplementation.test_parse_valid_yaml) ... ok
test_validate_rejects_unsupported_type (__main__.TestYamlImplementation.test_validate_rejects_unsupported_type) ... ok
test_write_read_round_trip (__main__.TestYamlImplementation.test_write_read_round_trip) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.012s

OK


Passed=True


## 7. Add Unit Tests for Every Implemented Abstract Method

The tests below cover:

- parsing valid YAML
- parsing malformed YAML
- loading from string, bytes, and streams
- serializing valid values
- writing to disk
- validating unsupported types
- round-trip integrity

In [13]:
# Validation examples
print(example.validate({'alpha': 1, 'beta': [True, False, 'x']}))

try:
    example.validate({'bad': object()})
except Exception as exc:
    print(type(exc).__name__, exc)

try:
    example.read('missing.yaml')
except Exception as exc:
    print(type(exc).__name__, exc)

print('schema default:', example.schema)
print('encoding default:', example.encoding)


{'alpha': 1, 'beta': [True, False, 'x']}
TypeError Unsupported YAML value type: <class 'object'>
FileNotFoundError YAML file does not exist: missing.yaml
schema default: safe
encoding default: utf-8


## 6. Implement Validation, Error Handling, and Defaults

This is where a real YAML class becomes useful in production: it should validate incoming values, normalize defaults, and raise clear exceptions instead of failing with unhelpful stack traces.

In [14]:
doc = {'title': 'example', 'items': ['a', 'b', 'c'], 'enabled': True}

serialized = example.dump(doc, sort_keys=True)
print(serialized)

path = Path('demo_output.yaml')
example.write(path, doc)
print(path.read_text())

roundtrip = example.parse(path.read_text())
print(roundtrip == doc)

path.unlink(missing_ok=True)


enabled: true
items:
- a
- b
- c
title: example

enabled: true
items:
- a
- b
- c
title: example

True


## 5. Implement Abstract Write/Serialize Methods

This section covers the serialization path:

- Python object to YAML string
- deterministic ordering when requested
- writing to a file or path
- preserving data structure safely and predictably

In [15]:
example = yaml()

parsed = example.parse(SAMPLE_YAML)
print(parsed)
print(type(parsed))

stream = io.StringIO(SAMPLE_YAML)
loaded_from_stream = example.load(stream)
print(loaded_from_stream == parsed)

from_bytes = example.load(SAMPLE_YAML.encode('utf-8'))
print(from_bytes == parsed)

try:
    example.parse(INVALID_YAML)
except Exception as exc:
    print(type(exc).__name__, exc)


{'name': 'demo', 'enabled': True, 'count': 3, 'items': ['one', 'two', 'three']}
<class 'dict'>
True
True
YamlError Unable to parse YAML: while parsing a flow sequence
  in "<unicode string>", line 1, column 7:
    name: [broken
          ^
expected ',' or ']', but got '<stream end>'
  in "<unicode string>", line 1, column 14:
    name: [broken
                 ^


## 4. Implement Abstract Read/Parse Methods

The parsing implementation handles:

- string input
- bytes input
- file or stream input
- safe parsing vs optional schema selection
- malformed YAML with clear exceptions

In [16]:
class YamlError(ValueError):
    """Raised when YAML parsing, validation, or I/O fails."""


class yaml(YamlContract):
    def __init__(self, *, schema: str = 'safe', encoding: str = 'utf-8', strict: bool = True):
        self.schema = schema
        self.encoding = encoding
        self.strict = strict

    @classmethod
    def parse(cls, source: str, *, schema: str = 'safe') -> Any:
        if not isinstance(source, str):
            raise TypeError('source must be a string')
        if not source.strip():
            return None
        try:
            if schema == 'safe':
                return pyyaml.safe_load(source)
            if schema == 'full':
                return pyyaml.load(source, Loader=pyyaml.FullLoader)
            raise ValueError(f'Unsupported schema: {schema!r}')
        except pyyaml.YAMLError as exc:
            raise YamlError(f'Unable to parse YAML: {exc}') from exc

    def load(self, source: Any, *, schema: str = 'safe') -> Any:
        if isinstance(source, (str, bytes)):
            text = source.decode('utf-8') if isinstance(source, bytes) else source
            return self.parse(text, schema=schema)
        if hasattr(source, 'read'):
            content = source.read()
            if isinstance(content, bytes):
                content = content.decode(self.encoding)
            return self.parse(content, schema=schema)
        if source is None:
            return None
        raise TypeError('load() expects a YAML string, bytes, stream, or None')

    def dump(self, value: Any, *, sort_keys: bool = True, default_flow_style: bool = False) -> str:
        validated = self.validate(value)
        return pyyaml.safe_dump(
            validated,
            sort_keys=sort_keys,
            default_flow_style=default_flow_style,
        )

    def read(self, path: str | Path, *, encoding: str = 'utf-8', schema: str = 'safe') -> Any:
        file_path = Path(path)
        if not file_path.exists():
            raise FileNotFoundError(f'YAML file does not exist: {file_path}')
        return self.load(file_path.read_text(encoding=encoding), schema=schema)

    def write(self, path: str | Path, value: Any, *, encoding: str = 'utf-8', sort_keys: bool = True) -> None:
        file_path = Path(path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        text = self.dump(value, sort_keys=sort_keys)
        file_path.write_text(text, encoding=encoding)

    def validate(self, value: Any) -> Any:
        if value is None:
            return None
        if isinstance(value, (str, int, float, bool)):
            return value
        if isinstance(value, Mapping):
            for key, item in value.items():
                if not isinstance(key, (str, int)):
                    raise TypeError('Mapping keys must be strings or ints')
                value[key] = self.validate(item)
            return value
        if isinstance(value, Sequence) and not isinstance(value, (str, bytes, bytearray)):
            return [self.validate(item) for item in value]
        if isinstance(value, set):
            return [self.validate(item) for item in sorted(value, key=repr)]
        raise TypeError(f'Unsupported YAML value type: {type(value)!r}')


print('Concrete class created:', yaml)
print('Abstract methods still unresolved:', yaml.__abstractmethods__)


Concrete class created: <class '__main__.yaml'>
Abstract methods still unresolved: frozenset()


## 3. Create the Concrete `yaml` Class Skeleton

We will define a practical `yaml` implementation that inherits the abstract contract and keeps runtime defaults explicit.

The constructor sets a default schema and option set, while the methods themselves handle file IO, parsing, serialization, and validation.

In [17]:
class YamlContract(ABC):
    @classmethod
    @abstractmethod
    def parse(cls, source: str, *, schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def load(self, source: Any, *, schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def dump(self, value: Any, *, sort_keys: bool = True, default_flow_style: bool = False) -> str:
        raise NotImplementedError

    @abstractmethod
    def read(self, path: str | Path, *, encoding: str = 'utf-8', schema: str = 'safe') -> Any:
        raise NotImplementedError

    @abstractmethod
    def write(self, path: str | Path, value: Any, *, encoding: str = 'utf-8', sort_keys: bool = True) -> None:
        raise NotImplementedError

    @abstractmethod
    def validate(self, value: Any) -> Any:
        raise NotImplementedError


print('Abstract methods:', YamlContract.__abstractmethods__)


Abstract methods: frozenset({'read', 'parse', 'load', 'validate', 'dump', 'write'})


## 2. Inspect the Abstract Contract for `yaml`

We define a small abstract base class and check which methods are still abstract before implementing them.

This gives us a clear contract: what each method accepts, returns, and what errors should be raised when the input is malformed.

## 1. Set Up Project Dependencies and Test Fixtures

This section installs/loads the required YAML library and defines input fixtures used throughout the notebook.

The examples intentionally use a clearly named custom `yaml` class so we can inspect the abstract contract and then implement it in a practical way.

In [18]:
import io
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional, Sequence

try:
    import yaml as pyyaml
except Exception as exc:  # pragma: no cover
    raise RuntimeError('PyYAML is required for this notebook.') from exc

# Test fixtures
SAMPLE_YAML = '''
name: demo
enabled: true
count: 3
items:
  - one
  - two
  - three
'''

VALID_DOC = {
    'name': 'demo',
    'enabled': True,
    'count': 3,
    'items': ['one', 'two', 'three'],
}

INVALID_YAML = 'name: [broken'


# Implementing Unimplemented Abstract Methods for a YAML Class

This notebook demonstrates how to turn an abstract YAML API into a practical concrete implementation with parsing, serialization, validation, and round-trip behavior.

## Goal

- inspect the abstract contract for a `yaml` class
- implement all abstract methods with useful logic
- validate behavior with focused tests
- ensure odd edge cases and malformed input are handled clearly